# Advanced RAG Techniques

## Why Advanced RAG?
Basic RAG fails when:
- Query is ambiguous or vague
- Relevant chunks are scattered
- Top-k retrieval returns redundant results
- Retrieved context doesn't match the question closely enough

## Advanced Techniques Overview

```
Pre-retrieval:    Query rewriting, HyDE, multi-query
Retrieval:        Hybrid search, re-ranking, parent-doc retriever
Post-retrieval:   Contextual compression, RAG fusion
Self-correcting:  Self-RAG, Corrective RAG, Agentic RAG
Graph-enhanced:   GraphRAG
```

## RAG Fusion Reciprocal Rank Fusion

Generate multiple queries, retrieve for each, then merge using RRF:

$$RRF(d) = \sum_{r \in R} \frac{1}{k + r(d)}$$

where $k=60$ (constant), $r(d)$ is the rank of document $d$ in result list $r$.

## Hybrid Search

Combine sparse (BM25) and dense (embedding) retrieval:

$$\text{score}(d,q) = \alpha \cdot \text{dense\_score}(d,q) + (1-\alpha) \cdot \text{sparse\_score}(d,q)$$

In [1]:
import numpy as np
from collections import defaultdict

# --- 1. Reciprocal Rank Fusion ---
def reciprocal_rank_fusion(ranked_lists, k=60):
    """
    Merge multiple ranked lists using RRF.
    ranked_lists: list of lists of doc_ids in ranked order
    """
    scores = defaultdict(float)
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked, start=1):
            scores[doc_id] += 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# Example
list1 = ['doc_A', 'doc_B', 'doc_C', 'doc_D']
list2 = ['doc_C', 'doc_A', 'doc_E', 'doc_B']
list3 = ['doc_B', 'doc_C', 'doc_A', 'doc_F']

fused = reciprocal_rank_fusion([list1, list2, list3])
print('RRF Merged Rankings:')
for doc, score in fused:
    print(f'  {doc}: {score:.4f}')

RRF Merged Rankings:
  doc_A: 0.0484
  doc_C: 0.0484
  doc_B: 0.0481
  doc_E: 0.0159
  doc_D: 0.0156
  doc_F: 0.0156


In [2]:
# --- 2. Multi-Query Retrieval with LangChain ---
MULTI_QUERY_CODE = '''
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Automatically generates 3 variations of the original query
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)

# Original query: "What is RAG?"
# Generated queries might be:
#   1. "Explain retrieval-augmented generation"
#   2. "How does RAG work in NLP?"
#   3. "What are the components of a RAG system?"
docs = retriever.get_relevant_documents("What is RAG?")
'''
print(MULTI_QUERY_CODE)


from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Automatically generates 3 variations of the original query
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)

# Original query: "What is RAG?"
# Generated queries might be:
#   1. "Explain retrieval-augmented generation"
#   2. "How does RAG work in NLP?"
#   3. "What are the components of a RAG system?"
docs = retriever.get_relevant_documents("What is RAG?")



In [3]:
# --- 3. HyDE (Hypothetical Document Embeddings) ---
# Instead of embedding the query directly, generate a hypothetical answer
# then embed that answer for retrieval

HYDE_CODE = '''
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser

hyde_prompt = ChatPromptTemplate.from_template("""
Write a short paragraph that would answer this question.
Do not say you don't know hypothesize an answer.
Question: {question}
Hypothetical answer:
""")

hyde_chain = hyde_prompt | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()

def hyde_retrieve(question, retriever):
    # Generate hypothetical document
    hyp_doc = hyde_chain.invoke({"question": question})
    print(f"Hypothetical doc: {hyp_doc[:100]}...")
    # Retrieve using the hypothetical doc as query
    return retriever.get_relevant_documents(hyp_doc)
'''
print(HYDE_CODE)


from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser

hyde_prompt = ChatPromptTemplate.from_template("""
Write a short paragraph that would answer this question.
Do not say you don't know hypothesize an answer.
Question: {question}
Hypothetical answer:
""")

hyde_chain = hyde_prompt | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()

def hyde_retrieve(question, retriever):
    # Generate hypothetical document
    hyp_doc = hyde_chain.invoke({"question": question})
    print(f"Hypothetical doc: {hyp_doc[:100]}...")
    # Retrieve using the hypothetical doc as query
    return retriever.get_relevant_documents(hyp_doc)



In [4]:
# --- 4. BM25 Sparse Retrieval ---
# pip install rank_bm25
from rank_bm25 import BM25Okapi

corpus = [
    "RAG combines retrieval with generation in language models",
    "Vector databases store embeddings for fast similarity search",
    "FAISS is a library for efficient similarity search",
    "Fine-tuning adapts a pretrained model to a specific task",
    "Retrieval systems find relevant documents from large corpora",
]

tokenized = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized)

query = "vector similarity search"
scores = bm25.get_scores(query.lower().split())
top_n = np.argsort(scores)[::-1][:3]
print('BM25 Results:')
for i in top_n:
    print(f'  [{scores[i]:.3f}] {corpus[i]}')

BM25 Results:
  [1.791] Vector databases store embeddings for fast similarity search
  [0.680] FAISS is a library for efficient similarity search
  [0.000] Retrieval systems find relevant documents from large corpora


In [5]:
# --- 5. Cross-Encoder Re-ranking ---
# Cross-encoders process (query, doc) pairs together more accurate but slower
# Use as a re-ranker after initial retrieval

RERANKER_CODE = '''
from sentence_transformers import CrossEncoder

# Load cross-encoder
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, docs, top_k=3):
    pairs = [(query, doc) for doc in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# Also available: Cohere Reranker (API-based)
# import cohere
# co = cohere.Client("API_KEY")
# results = co.rerank(query=query, documents=docs, model="rerank-english-v3.0", top_n=3)
'''
print(RERANKER_CODE)


from sentence_transformers import CrossEncoder

# Load cross-encoder
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, docs, top_k=3):
    pairs = [(query, doc) for doc in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# Also available: Cohere Reranker (API-based)
# import cohere
# co = cohere.Client("API_KEY")
# results = co.rerank(query=query, documents=docs, model="rerank-english-v3.0", top_n=3)



In [6]:
# --- 6. Self-RAG Pattern ---
# Model decides whether to retrieve, then reflects on retrieved content

SELF_RAG_CODE = '''
# Self-RAG uses special tokens:
# [Retrieve] should we retrieve?
# [IsRel]    is the retrieved doc relevant?
# [IsSup]    does the doc support the generation?
# [IsUse]    is the response useful?

# Simplified implementation using prompting
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini")

def self_rag(question, retriever):
    # Step 1: Should we retrieve?
    retrieve_prompt = f"Does answering '{question}' require external knowledge? Reply YES or NO only."
    need_retrieve = llm.invoke(retrieve_prompt).content.strip().upper() == "YES"

    if need_retrieve:
        docs = retriever.get_relevant_documents(question)
        context = "\\n".join(d.page_content for d in docs)

        # Step 2: Is retrieved content relevant?
        rel_prompt = f"Is this context relevant to '{question}'?\\nContext: {context[:200]}\\nAnswer YES or NO."
        is_relevant = llm.invoke(rel_prompt).content.strip().upper() == "YES"

        if is_relevant:
            answer_prompt = f"Answer based on context:\\n{context}\\n\\nQuestion: {question}"
        else:
            answer_prompt = question  # Fall back to parametric knowledge
    else:
        answer_prompt = question

    return llm.invoke(answer_prompt).content
'''
print(SELF_RAG_CODE)


# Self-RAG uses special tokens:
# [Retrieve] should we retrieve?
# [IsRel]    is the retrieved doc relevant?
# [IsSup]    does the doc support the generation?
# [IsUse]    is the response useful?

# Simplified implementation using prompting
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini")

def self_rag(question, retriever):
    # Step 1: Should we retrieve?
    retrieve_prompt = f"Does answering '{question}' require external knowledge? Reply YES or NO only."
    need_retrieve = llm.invoke(retrieve_prompt).content.strip().upper() == "YES"

    if need_retrieve:
        docs = retriever.get_relevant_documents(question)
        context = "\n".join(d.page_content for d in docs)

        # Step 2: Is retrieved content relevant?
        rel_prompt = f"Is this context relevant to '{question}'?\nContext: {context[:200]}\nAnswer YES or NO."
        is_relevant = llm.invoke(rel_prompt).content.strip().upper() ==

In [7]:
# --- 7. RAGAS Evaluation ---
# pip install ragas

RAGAS_CODE = '''
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

# Prepare evaluation dataset
eval_data = {
    "question": ["What is RAG?", "What are vector databases?"],
    "answer": ["RAG combines retrieval with LLM generation.", "Vector databases store and search embeddings."],
    "contexts": [
        ["RAG stands for Retrieval-Augmented Generation..."],
        ["Vector databases like FAISS, Chroma, Pinecone..."]
    ],
    "ground_truth": ["RAG is a technique combining retrieval systems with LLMs.", "Vector databases are specialized for embedding storage."]
}
dataset = Dataset.from_dict(eval_data)

# Evaluate
results = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
)
print(results)
# Faithfulness: is the answer grounded in the context?
# Answer Relevancy: does the answer address the question?
# Context Precision: are retrieved contexts relevant?
# Context Recall: are all required contexts retrieved?
'''
print(RAGAS_CODE)


from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

# Prepare evaluation dataset
eval_data = {
    "question": ["What is RAG?", "What are vector databases?"],
    "answer": ["RAG combines retrieval with LLM generation.", "Vector databases store and search embeddings."],
    "contexts": [
        ["RAG stands for Retrieval-Augmented Generation..."],
        ["Vector databases like FAISS, Chroma, Pinecone..."]
    ],
    "ground_truth": ["RAG is a technique combining retrieval systems with LLMs.", "Vector databases are specialized for embedding storage."]
}
dataset = Dataset.from_dict(eval_data)

# Evaluate
results = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
)
print(results)
# Faithfulness: is the answer grounded in the context?
# Answer Relevancy: does the answer address the question?
# Context Precision: are retrieved contexts re

## Additional Learning Resources

### Papers
- [Self-RAG](https://arxiv.org/abs/2310.11511) Asai et al., 2023
- [Corrective RAG (CRAG)](https://arxiv.org/abs/2401.15884) Yan et al., 2024
- [RAG Fusion](https://arxiv.org/abs/2402.03367)
- [HyDE](https://arxiv.org/abs/2212.10496) Gao et al., 2022
- [RAGAS Evaluation](https://arxiv.org/abs/2309.15217)
- [RAG Survey](https://arxiv.org/abs/2312.10997) Gao et al., 2023

### Tools & Docs
- [LangChain Advanced RAG](https://python.langchain.com/docs/tutorials/rag/)
- [LlamaIndex Advanced Retrieval](https://docs.llamaindex.ai/en/stable/module_guides/querying/retriever/)
- [RAGAS GitHub](https://github.com/explodinggradients/ragas)
- [Pinecone RAG Series](https://www.pinecone.io/learn/series/rag/)

### Videos
- [Advanced RAG James Briggs](https://www.youtube.com/watch?v=JChPi0BR3cw)
- [RAG from Scratch LangChain](https://www.youtube.com/playlist?list=PLfaIDFEXuae2LXb3FO0Q9KxxmNa1efnKi)